# 15SW Safety Forecast — the scoring "brain"

This notebook is the **offline model** for the *This Week — Safety Forecast* panel.

**How it fits the app**

```
THIS NOTEBOOK (run weekly)                         THE APP (always on)
  records + METAR weather + (flying hours later)     reads ONE row and
        -> features -> model -> risk + reasons  ──►  shows it in plain words
                    writes to safety_forecasts table
```

The app does **no calculation**. It only reads the latest row this notebook writes.

**Honesty notes**
- ~128 mishaps over 11 years = *rare events, small data*. We keep to a few strong
  features and a penalized model. No deep learning (it would overfit).
- Until real **flying hours** are added as exposure, this is an **indicative risk
  score**, not a true per-flight probability. The upgrade path is noted at the end.


## 0. Requirements

Run once in your Python environment:

```
pip install pandas numpy requests scikit-learn
```

Everything below is organised in small steps so you can run and inspect them one
at a time.


In [1]:
# --- Configuration -------------------------------------------------------
from pathlib import Path
import sqlite3, json, re
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Path to the app's SQLite database. Tries a few locations so the notebook works
# whether it's run from the notebooks/ folder or the project root.
def _find_db():
    for p in [Path("../database/database.sqlite"), Path("database/database.sqlite"),
              Path.cwd() / "database/database.sqlite"]:
        if p.exists():
            return p.resolve()
    return Path("../database/database.sqlite").resolve()
DB_PATH = _find_db()

# The two home bases (ICAO codes on the IEM METAR archive).
STATIONS = {
    "Sangley / MDAAB / Cavite": "RPLS",
    "Lumbia / Cagayan de Oro":  "RPML",
}

# Pull live METAR from the internet? If False, the notebook still runs using a
# neutral (no-weather) fallback so you can develop offline.
USE_LIVE_METAR = True

# Current ENSO advisory for recent months that NOAA's ONI table hasn't published
# yet. Set from the latest PAGASA/NOAA bulletin: "el_nino", "la_nina", or None.
RECENT_ENSO_OVERRIDE = "el_nino"

print("DB:", DB_PATH, "| exists:", DB_PATH.exists())


DB: C:\Users\User\Desktop\safety\database\database.sqlite | exists: True


## 1. Load the mishap records

We read straight from the app's database so the model always uses the live
record. Only the columns we need.

In [2]:
def load_records(db_path: Path) -> pd.DataFrame:
    con = sqlite3.connect(db_path)
    try:
        df = pd.read_sql_query(
            "SELECT mishap_date, location, mishap_type, environment, category, description "
            "FROM mishaps", con, parse_dates=["mishap_date"])
    finally:
        con.close()
    df["year"] = df["mishap_date"].dt.year
    return df

records = load_records(DB_PATH)
print(f"{len(records)} records, {records.year.min()}-{records.year.max()}")
records.head()


128 records, 2016-2026


,mishap_date,location,mishap_type,environment,category,description,year
0,2016-02-04,MDAAB,incident,flight,Canopy / door detachment,"AW-109 Attack Helicopter with tail number 824,...",2016
1,2016-03-19,MDAAB,incident,flight,Bird / wildlife strike,AW-109AH # 822 encountered Bird Strike while p...,2016
2,2016-03-29,EAAB,incident,flight,Landing gear / tire / brake,OV-10 # 145 experienced blown tire during land...,2016
3,2016-04-18,EAAB,incident,flight,Landing gear / tire / brake,OV-10 # 636 experienced stuck-up RH Brake duri...,2016
4,2016-05-04,MDAAB,incident,flight,Bird / wildlife strike,The Horizontal Stabilizer of OV-10 #636 encoun...,2016


## 2. Extract tail number + aircraft type from the description

Your descriptions contain tail numbers ("A-29B ST Nr 1905", "MD-520MG #505").
Pulling these out now means that when your **per-tail flying hours** arrive, they
join on `tail x week` with zero rework — and we get aircraft-type features today.

In [3]:
AIRCRAFT_TYPES = ["OV-10", "A-29B", "A29B", "SF-260", "T-129", "AW-109", "AW109",
                   "MD-520", "MD520", "MD-500", "MD500"]

def find_aircraft(text: str):
    t = str(text)
    for a in AIRCRAFT_TYPES:
        if a.lower() in t.lower():
            return a.replace("A29B", "A-29B").replace("AW109", "AW-109") \
                    .replace("MD520", "MD-520").replace("MD500", "MD-500")
    return None

def find_tail(text: str):
    # e.g. "# 824", "#505", "Nr 1905", "number 824"
    m = re.search(r"(?:#|nr\.?|number)\s*([0-9]{3,4})", str(text), flags=re.I)
    return m.group(1) if m else None

records["aircraft"] = records["description"].map(find_aircraft)
records["tail"] = records["description"].map(find_tail)
records[["mishap_date", "environment", "aircraft", "tail"]].head(10)


,mishap_date,environment,aircraft,tail
0,2016-02-04,flight,AW-109,824
1,2016-03-19,flight,AW-109,822
2,2016-03-29,flight,OV-10,145
3,2016-04-18,flight,OV-10,636
4,2016-05-04,flight,OV-10,636
5,2016-05-04,flight,SF-260,701
6,2016-05-26,flight,AW-109,819
7,2016-05-26,ground,NaN,NaN
8,2016-07-10,ground,NaN,NaN
9,2016-07-26,flight,MD-520,438


## 3. Pull METAR weather (visibility, haze, wind)

METAR is the *aviation* weather record: per-airport, hourly, with **visibility**
and a present-weather code where **`HZ` = haze** (your example). Source: the free
Iowa State IEM archive (no account needed).

If `USE_LIVE_METAR` is False or the network is unavailable, we fall back to a
neutral frame so the rest of the notebook still runs.

In [4]:
import requests

def fetch_metar(station: str, start: datetime, end: datetime) -> pd.DataFrame:
    url = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"
    params = {
        "station": station, "data": ["vsby", "sknt", "gust", "wxcodes"],
        "year1": start.year, "month1": start.month, "day1": start.day,
        "year2": end.year, "month2": end.month, "day2": end.day,
        "tz": "Etc/UTC", "format": "onlycomma", "missing": "M", "trace": "T",
        "latlon": "no",
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    from io import StringIO
    df = pd.read_csv(StringIO(r.text), na_values=["M", "T"])
    df["valid"] = pd.to_datetime(df["valid"])
    df["station"] = station
    return df

start = datetime(int(records.year.min()), 1, 1)
end = datetime.utcnow()

metar_frames = []
if USE_LIVE_METAR:
    for label, st in STATIONS.items():
        try:
            f = fetch_metar(st, start, end)
            metar_frames.append(f)
            print(f"{st}: {len(f)} obs")
        except Exception as e:
            print(f"{st}: FAILED ({e})")

metar = pd.concat(metar_frames, ignore_index=True) if metar_frames else pd.DataFrame()
print("total METAR rows:", len(metar))


C:\Users\User\AppData\Local\Temp\ipykernel_24036\2500888738.py:21: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


RPLS: 0 obs


RPML: 0 obs
total METAR rows: 0


## 4. Summarise weather to one row per week + station

We reduce raw hourly METAR to weekly numbers we can use as features:
- **avg / min visibility** (miles)
- **haze hours** (present-weather contains `HZ`)
- **max wind / gust** (knots)

In [5]:
ISO = lambda d: pd.to_datetime(d).dt.to_period("W-SUN").apply(lambda p: p.start_time.date())

def weekly_weather(metar: pd.DataFrame) -> pd.DataFrame:
    if metar.empty:
        return pd.DataFrame(columns=["week", "station", "vsby_avg", "vsby_min",
                                     "haze_hours", "wind_max"])
    m = metar.copy()
    m["week"] = ISO(m["valid"])
    m["is_haze"] = m["wxcodes"].fillna("").str.contains("HZ")
    for c in ["vsby", "sknt", "gust"]:
        m[c] = pd.to_numeric(m[c], errors="coerce")
    g = m.groupby(["week", "station"]).agg(
        vsby_avg=("vsby", "mean"), vsby_min=("vsby", "min"),
        haze_hours=("is_haze", "sum"),
        wind_max=("sknt", "max"),
    ).reset_index()
    return g

wx = weekly_weather(metar)
wx.head()


,week,station,vsby_avg,vsby_min,haze_hours,wind_max


## 5. Build the weekly panel

One row per **week x base x environment**, with:
- `mishaps` — the target (count that week)
- weather summary (from step 4)
- **season** features: Habagat (SW monsoon, Jun-Sep), Amihan (NE, Nov-Apr),
  and a **bird-migration** window (Sep-Nov, tune to your history)
- **ENSO / El Nino** flag from NOAA's ONI (drier, hazier climate regime) — a
  multi-year climate state, distinct from the intra-year monsoon
- **recent_mishaps** — count in the previous 4 weeks (momentum)
- `flying_hours` — **stubbed (NaN)** until your per-tail hours arrive

In [6]:
def fetch_oni():
    # NOAA CPC Oceanic Nino Index (ONI): 3-month running SST anomaly. >= +0.5 is
    # El Nino, <= -0.5 is La Nina. Returns {(year, center_month): anomaly}.
    url = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"
    center = {"DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4, "AMJ": 5, "MJJ": 6,
              "JJA": 7, "JAS": 8, "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12}
    lookup = {}
    try:
        for line in requests.get(url, timeout=60).text.splitlines()[1:]:
            p = line.split()
            if len(p) >= 4 and p[0] in center:
                lookup[(int(p[1]), center[p[0]])] = float(p[3])
    except Exception as e:
        print("ONI fetch failed (using override/neutral):", e)
    return lookup

ONI = fetch_oni()
print("ONI months loaded:", len(ONI))

def enso_state(year, month):
    # -> (el_nino, la_nina) flags. Recent unpublished months use the analyst
    # override; otherwise carry the most recent known ONI forward.
    v = ONI.get((year, month))
    if v is None:
        if RECENT_ENSO_OVERRIDE == "el_nino": return 1, 0
        if RECENT_ENSO_OVERRIDE == "la_nina": return 0, 1
        v = ONI[max(ONI)] if ONI else 0.0
    return int(v >= 0.5), int(v <= -0.5)

def season(week_date):
    mth = pd.to_datetime(week_date).month
    if mth in (6, 7, 8, 9):   return "habagat"   # SW monsoon (rainy)
    if mth in (11, 12, 1, 2, 3, 4): return "amihan"  # NE monsoon (dry/cool)
    return "transition"

def is_bird_season(week_date):
    return int(pd.to_datetime(week_date).month in (9, 10, 11))

def build_panel(records: pd.DataFrame) -> pd.DataFrame:
    r = records.dropna(subset=["mishap_date"]).copy()
    r["week"] = ISO(r["mishap_date"])
    # full weekly calendar so weeks with zero mishaps are represented (correct
    # for rare-event modelling)
    weeks = pd.period_range(r["mishap_date"].min(), pd.Timestamp.utcnow(), freq="W-SUN")
    weeks = [p.start_time.date() for p in weeks]
    envs = ["flight", "ground"]
    grid = pd.MultiIndex.from_product([weeks, envs], names=["week", "environment"]).to_frame(index=False)

    counts = r.groupby(["week", "environment"]).size().rename("mishaps").reset_index()
    panel = grid.merge(counts, on=["week", "environment"], how="left").fillna({"mishaps": 0})
    panel["mishaps"] = panel["mishaps"].astype(int)

    panel["season"] = panel["week"].map(season)
    panel["habagat"] = (panel["season"] == "habagat").astype(int)  # SW monsoon flag
    panel["bird_season"] = panel["week"].map(is_bird_season)
    panel["month"] = pd.to_datetime(panel["week"]).dt.month
    panel["year"] = pd.to_datetime(panel["week"]).dt.year
    enso = panel.apply(lambda r: enso_state(int(r["year"]), int(r["month"])), axis=1)
    panel["el_nino"] = [e[0] for e in enso]   # dry/hazy regime
    panel["la_nina"] = [e[1] for e in enso]   # wet/stormy regime

    # recent momentum: mishaps in the previous 4 weeks (same environment)
    panel = panel.sort_values(["environment", "week"])
    panel["recent_mishaps"] = (panel.groupby("environment")["mishaps"]
                               .transform(lambda s: s.shift(1).rolling(4, min_periods=1).sum())
                               .fillna(0))

    # weather: average the stations for now (per-base join is a later refinement)
    if not wx.empty:
        wwk = wx.groupby("week").agg(vsby_min=("vsby_min", "min"),
                                     haze_hours=("haze_hours", "sum"),
                                     wind_max=("wind_max", "max")).reset_index()
        panel = panel.merge(wwk, on="week", how="left")
    for c in ["vsby_min", "haze_hours", "wind_max"]:
        if c not in panel: panel[c] = np.nan

    panel["flying_hours"] = np.nan   # <-- exposure stub; fill when hours arrive
    return panel

panel = build_panel(records)
print(panel.shape)
panel.tail(8)


ONI months loaded: 919
(1108, 15)


C:\Users\User\AppData\Local\Temp\ipykernel_24036\3538036355.py:44: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  weeks = pd.period_range(r["mishap_date"].min(), pd.Timestamp.utcnow(), freq="W-SUN")


,week,environment,mishaps,season,habagat,bird_season,month,year,el_nino,la_nina,recent_mishaps,vsby_min,haze_hours,wind_max,flying_hours
1093,2026-07-20,ground,0,habagat,1,0,7,2026,1,0,0.0,NaN,NaN,NaN,NaN
1095,2026-07-27,ground,0,habagat,1,0,7,2026,1,0,0.0,NaN,NaN,NaN,NaN
1097,2026-08-03,ground,0,habagat,1,0,8,2026,1,0,0.0,NaN,NaN,NaN,NaN
1099,2026-08-10,ground,0,habagat,1,0,8,2026,1,0,0.0,NaN,NaN,NaN,NaN
1101,2026-08-17,ground,0,habagat,1,0,8,2026,1,0,0.0,NaN,NaN,NaN,NaN
1103,2026-08-24,ground,0,habagat,1,0,8,2026,1,0,0.0,NaN,NaN,NaN,NaN
1105,2026-08-31,ground,0,habagat,1,0,8,2026,1,0,0.0,NaN,NaN,NaN,NaN
1107,2026-09-07,ground,0,habagat,1,1,9,2026,1,0,0.0,NaN,NaN,NaN,NaN


## 6. Interim model — penalized logistic regression

We predict **"a mishap happens this week (per environment)"** from the features.
Choices made deliberately for small/rare data:
- **Logistic regression with L2 penalty** and `class_weight="balanced"` — stable,
  interpretable, each coefficient is a risk weight (maps onto a FRAT card).
- We use the **Flight** rows for the flagship model (the ground side has no clean
  exposure). You can repeat for ground as a relative-risk indicator.

> **Upgrade path:** once `flying_hours` is populated, switch to a **Poisson/NB
> regression with `log(flying_hours)` as an offset** to get true *mishaps per
> 1,000 flying hours*. Same features, one extra term.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

FEATURES = ["el_nino", "habagat", "bird_season", "recent_mishaps", "vsby_min", "haze_hours", "wind_max"]

def train(panel: pd.DataFrame, environment="flight"):
    d = panel[panel["environment"] == environment].copy()
    d[FEATURES] = d[FEATURES].fillna(d[FEATURES].median(numeric_only=True))
    # if weather is entirely missing (offline), those columns are constant -> fine
    d = d.fillna(0)
    y = (d["mishaps"] > 0).astype(int)
    X = d[FEATURES]
    if y.nunique() < 2:
        print("Not enough signal to train; returning None.")
        return None, None
    model = make_pipeline(StandardScaler(),
                          LogisticRegression(penalty="l2", C=0.5,
                                             class_weight="balanced", max_iter=1000))
    model.fit(X, y)
    return model, d

model, flight_panel = train(panel, "flight")
model


C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['el_nino','habagat','bird_season',...,'vsby_min','haze_hours','wind_max']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


## 7. Score this week + translate to plain language

We take the current week's features, get the model's probability, map it to a
**band** (Low / Moderate / Elevated / High), and turn each feature into a plain
sentence — "bad" / "unusual" means *worse than the historical norm*.

In [8]:
def band_from_prob(p):
    if p >= 0.66: return "high"
    if p >= 0.40: return "elevated"
    if p >= 0.20: return "moderate"
    return "low"

def hist_week(records, week_date):
    # How many mishaps historically fall in this same calendar week (any year).
    start_doy = pd.to_datetime(week_date).dayofyear
    doy = records["mishap_date"].dt.dayofyear
    sub = records[((doy - start_doy) % 365) <= 6]
    return len(sub), int(sub["mishap_date"].dt.year.nunique())

def build_factors(row, d, top=5):
    # FRAT-style: the risk conditions actually present this week, each with a
    # transparent weight (0-100) for the bar. The overall %/band still comes from
    # the model; this panel explains *what is raising the risk*.
    factors = []
    hc, hy = hist_week(records, row["week"])
    if hc > 0:
        factors.append((f"{hc} past mishaps in this calendar week (over {hy} yrs)", "info", min(35 + 18 * hc, 100)))
    if row.get("bird_season"):
        factors.append(("Bird-strike season active", "alert", 80))
    if row.get("el_nino"):
        factors.append(("El Nino - drier, hazier conditions", "alert", 70))
    elif row.get("la_nina"):
        factors.append(("La Nina - wetter, more storms", "info", 55))
    if row.get("habagat") and not row.get("el_nino"):
        factors.append(("Habagat (rainy) season", "info", 60))
    rec = int(row.get("recent_mishaps", 0) or 0)
    if rec >= 1:
        factors.append((f"{rec} recent mishap(s) in the last 4 weeks", "info", min(40 + 20 * rec, 100)))
    vm = row.get("vsby_min")
    if pd.notna(vm) and vm > 0 and vm < (d["vsby_min"].median() or 0) * 0.6:
        factors.append(("Low visibility / haze", "alert", 90))
    wm = row.get("wind_max")
    if pd.notna(wm) and wm > (d["wind_max"].median() or 0) + 8:
        factors.append(("Strong winds", "info", 50))
    if not factors:
        return [{"text": "Conditions look normal for this week.", "tone": "good", "impact": 0}]
    factors.sort(key=lambda t: -t[2])
    return [{"text": t, "tone": tone, "impact": imp} for t, tone, imp in factors[:top]]

def score_week(row, d, model):
    if model is not None:
        X = pd.DataFrame([row[FEATURES].fillna(0)])
        prob = float(model.predict_proba(X)[0, 1])
    else:
        prob = min(0.2 + 0.1 * row.get("recent_mishaps", 0), 0.9)  # fallback
    level = band_from_prob(prob)
    return {
        "week_start": str(row["week"]),
        "risk_level": level,
        "likelihood": int(round(prob * 100)),
        "headline": {
            "high": "High risk this week - brief crews and review go/no-go.",
            "elevated": "Elevated risk this week - brief crews before flying and driving.",
            "moderate": "Moderate risk this week - normal precautions.",
            "low": "Low risk this week.",
        }[level],
        "reasons": build_factors(row, d),
    }

flight_d = panel[panel["environment"] == "flight"].copy()
result = score_week(flight_d.sort_values("week").iloc[-1], flight_d, model)
result


{'week_start': '2026-09-07',
 'risk_level': 'moderate',
 'likelihood': 37,
 'headline': 'Moderate risk this week - normal precautions.',
 'reasons': [{'text': '3 past mishaps in this calendar week (over 2 yrs)',
   'tone': 'info',
   'impact': 89},
  {'text': 'Bird-strike season active', 'tone': 'alert', 'impact': 80},
  {'text': 'El Nino - drier, hazier conditions',
   'tone': 'alert',
   'impact': 70}]}

## 8. Write the result to the app

This is the **only** thing the app reads. We upsert **one wing-wide row per week
of the current year** into `safety_forecasts` — that feeds both the week-changer
and the "risk across the year" chart. Running the notebook again overwrites them.

In [9]:
def write_forecasts(db_path: Path, results: list, source="model v0"):
    now = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
    con = sqlite3.connect(db_path)
    try:
        cur = con.cursor()
        for res in results:
            # wing-wide row => base IS NULL; clear then insert (NULLs bypass UNIQUE)
            cur.execute("DELETE FROM safety_forecasts WHERE date(week_start)=date(?) AND base IS NULL",
                        (res["week_start"],))
            cur.execute(
                "INSERT INTO safety_forecasts (week_start, base, risk_level, likelihood, "
                "headline, reasons, source, generated_at, created_at, updated_at) "
                "VALUES (?, NULL, ?, ?, ?, ?, ?, ?, ?, ?)",
                (res["week_start"], res["risk_level"], res["likelihood"],
                 res["headline"], json.dumps(res["reasons"]), source, now, now, now))
        con.commit()
    finally:
        con.close()
    print(f"Wrote {len(results)} weekly forecasts")

# Score EVERY week of the current year so the dashboard's week-changer and the
# "risk across the year" chart have a full series to show.
this_year = datetime.utcnow().year
year_rows = flight_d[pd.to_datetime(flight_d["week"]).dt.year == this_year].sort_values("week")
results = [score_week(r, flight_d, model) for _, r in year_rows.iterrows()]

write_forecasts(DB_PATH, results)


Wrote 36 weekly forecasts


C:\Users\User\AppData\Local\Temp\ipykernel_24036\3348399920.py:23: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  this_year = datetime.utcnow().year
C:\Users\User\AppData\Local\Temp\ipykernel_24036\3348399920.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


## 9. Next steps

1. **Flying hours (exposure).** Add `flying_hours[tail, week]`, switch step 6 to
   Poisson/NB with `log(flying_hours)` offset -> true *mishaps per 1,000 hours*.
2. **Per-base weather join** instead of averaging stations.
3. **Extract more fields** (phase of flight, rank, time) - the local LLM can do
   this from the descriptions.
4. **Validate** with time-based cross-validation; report **AUC / Brier score** so
   the forecast's accuracy is stated honestly, not assumed.
5. **Schedule** this notebook (or a script export) to run weekly.
